In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import seaborn as sns
from localcider.sequenceParameters import SequenceParameters
import matplotlib.pyplot as plt
import Bio.SeqIO
import sys
#/Users/aelazar/scripts
sys.path.append('/Users/aelazar/Downloads/iupred2a/')
#import iupred2a_lib
#from pyaaisc import Aaindex
import joblib
MUNV940103 ={'A':1.08,'L':0.789,'R':0.976,'K':1.026,'N':1.197,'M':.812,'D':1.266,'F':0.685,
             'C':.733,'P':1.412,'Q':1.05,'S':0.987,'E':1.085,'T':0.784,'G':1.104,'W':0.755,'H':0.906,
             'Y':0.665,'I':0.583,'V':0.546}#from AAindex MUNV940103
from scipy import stats


In [ ]:
l = []
for i in enumerate(list('VSVWVLQS')):
    l.append({'pos':i[0],'val':MUNV940103[i[1]],'name':'HTR7'})
for i in enumerate(list('YEVHHQKLVFFA')):
    l.append({'pos':i[0],'val':MUNV940103[i[1]],'name':'APP'})

In [ ]:
sns.lineplot(x='pos',y='val',data=pd.DataFrame(l),hue='name')

In [ ]:
ll = pd.DataFrame(l)
ll[ll.name!='HTR7'].val.mean()

In [ ]:
import numpy as np
from matplotlib.patches import PathPatch

def adjust_box_widths(g, fac):
    """
    Adjust the widths of a seaborn-generated boxplot.
    """

    # iterating through Axes instances
    for ax in g.axes:

        # iterating through axes artists:
        for c in ax.get_children():

            # searching for PathPatches
            if isinstance(c, PathPatch):
                # getting current width of box:
                p = c.get_path()
                verts = p.vertices
                verts_sub = verts[:-1]
                xmin = np.min(verts_sub[:, 0])
                xmax = np.max(verts_sub[:, 0])
                xmid = 0.5*(xmin+xmax)
                xhalf = 0.5*(xmax - xmin)

                # setting new width of box
                xmin_new = xmid-fac*xhalf
                xmax_new = xmid+fac*xhalf
                verts_sub[verts_sub[:, 0] == xmin, 0] = xmin_new
                verts_sub[verts_sub[:, 0] == xmax, 0] = xmax_new

                # setting new width of median line
                for l in ax.lines:
                    if np.all(l.get_xdata() == [xmin, xmax]):
                        l.set_xdata([xmin_new, xmax_new])

In [ ]:
waltzdb = pd.read_csv('/Users/aelazar/Downloads/waltzdb_export.csv')#http://waltzdb.switchlab.org/ 

#waltzdb = pd.read_csv('/Users/aelazar/PycharmProjects/PhaseSep/classifier/waltzdb_export.csv')#http://waltzdb.switchlab.org/ 
waltzdb = waltzdb.drop_duplicates(subset=['Sequence'])
waltzdb =waltzdb.reset_index()

In [ ]:
waltzdb[waltzdb.Classification!='amyloid']

In [ ]:
waltzdb['UniProt ID']

In [ ]:
df = pd.DataFrame()
dff = pd.DataFrame()
for k in waltzdb.index:
    if k%500==0:
        print(k)
    seq = waltzdb.loc[k,'Sequence']
    if waltzdb.loc[k,'Classification']=='amyloid':
        classif = 1
    else:
        classif = 0 
    dff = dff.append({'classification':classif,'beta':np.sum([MUNV940103[i] for i in waltzdb.loc[k,'Sequence']])},ignore_index=True)
df = pd.concat([dff, pd.get_dummies(waltzdb['Sequence'].apply(lambda x:pd.Series(list(x))))], axis=1)#One-hot encodong 

In [ ]:
df['AAseq'] = df.index.map(dict(zip(waltzdb.index,waltzdb.Sequence)))
df['acc'] = df.index.map(dict(zip(waltzdb.index,waltzdb['UniProt ID'])))
df.to_csv('waltzdb_Random_Forest_input.csv')

In [ ]:
df = df.iloc[np.random.permutation(len(df))]#shuffling the data
df = df.reset_index()
df = df.drop(['index'],axis=1)
set(df['type'])

In [ ]:
fig = plt.figure(figsize=(3, 4))
sns.violinplot(hue="type", y="beta", data=df_t,split=1,fill=0,hue_order=['Non-Amyloid','Amyloid'],
              palette=['gray','r'])
plt.savefig('Delta_G_beta_paper_v2.svg')
# ax = sns.boxplot(x='type',y='beta',data=df_t,notch=True,palette=['#B4B5B4','#9D282B'],width=.5,linewidth=2,showfliers=False)
# ax = sns.stripplot(x="type", y="beta", data=df_t, color="0",alpha=0.1)

In [ ]:
df_t = df 
df_t.loc[df_t[df_t.classification==1].index,'type'] = 'Amyloid'
df_t.loc[df_t[df_t.classification==0].index,'type'] = 'Non-Amyloid'
print(stats.ttest_ind(df_t[(df_t['type']=='Amyloid')].beta,df_t[(df_t['type']=='Non-Amyloid')].beta))
fig = plt.figure(figsize=(5, 7))
ax = sns.boxplot(x='type',y='beta',data=df_t,notch=True,palette=['#B4B5B4','#9D282B'],width=.5,linewidth=2,showfliers=False)
ax = sns.stripplot(x="type", y="beta", data=df_t, color="0",alpha=0.1)
ax.set(xlabel=None)
#ax.text(0.5, 0.05,'Text Here', fontsize=9)
#plt.text(0.1, 0.9, 'AUC=.88', size=15)#, color='purple')
sns.despine(offset=10, trim=True)
sns.set(font_scale=1.5,style='white')
#plt.axhline(y=.6, linewidth=1,ls='--', color='k')
plt.ylabel(r'$\sum\Delta G_{\beta} $')#'Free energy in beta-strand conformation'
adjust_box_widths(fig, 1.4)
x1, x2 = 0, 1   # columns 'Sat' and 'Sun' (first column: 0, see plt.xticks())
y, h, col = df_t['beta'].max() + .1, .1, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "$p=10^{-112}$", ha='center', va='bottom', color=col)

plt.tight_layout()
#plt.savefig('Delta_G_beta_paper.png',dpi=1200)

In [ ]:
df

In [ ]:
from sklearn import preprocessing
from sklearn.model_selection import KFold
from sklearn.model_selection  import cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
labels = np.array(df['classification'])
features= df.drop(['classification','type'], axis = 1)
print(features)
feature_list = list(df.drop(['classification','type'], axis = 1).columns)
# Convert to numpy array
features =  np.array(df.drop(['classification','type'], axis = 1))
from sklearn.model_selection import train_test_split
# Split the data into training and testing sets
train_features, test_features, train_labels, test_labels = train_test_split(features, labels, test_size = .4, random_state = 42)
print('Training Features Shape:', train_features.shape)
print('Training Labels Shape:', train_labels.shape)
print('Testing Features Shape:', test_features.shape)
print('Testing Labels Shape:', test_labels.shape)
# Import the model we are using
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier,ExtraTreesRegressor
kfold = KFold(n_splits=10)
rf = RandomForestRegressor(n_estimators = 24, random_state = 2,max_features = 'auto', bootstrap = True,)
# Train the model on training data
model = rf.fit(train_features, train_labels);

importances = list(rf.feature_importances_)
#List of tuples with variable and importance
feature_importances = [(feature, round(importance, 2)) for feature, importance in zip(feature_list, importances)]
# Sort the feature importances by most important first
feature_importances = sorted(feature_importances, key = lambda x: x[1], reverse = True)
# Print out the feature and importances 
[print('Variable: {:20} Importance: {:.2f}'.format(*pair)) for pair in feature_importances[0:10]]


In [ ]:
from sklearn.model_selection import StratifiedKFold,KFold,ShuffleSplit#https://scikit-learn.org/stable/auto_examples/model_selection/plot_roc_crossval.html
from sklearn.metrics import auc
from sklearn.metrics import plot_roc_curve
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
cv = StratifiedKFold(n_splits=5)
#train_features, test_features, train_labels, test_labels = train_test_split(features, labels, test_size = .25, random_state = 42)
#rf = RandomForestClassifier(n_estimators = 500, random_state = 0,max_features = 'auto', bootstrap = True)
rf = RandomForestClassifier(n_estimators=700,min_samples_split=2,min_samples_leaf=1,max_features=3,max_depth=None,bootstrap=False)
tprs = []
aucs = []
mean_fpr = np.linspace(0, 1, 100)
fig, ax = plt.subplots(figsize=(7, 7))
precision_score_list = []
recall_score_list  = []
f1_score_list = []
accuracy_score_list =[]
confusion_matrix_list = []
test_index_list = []
#for i, (train, test) in enumerate(cv.split(features,labels)):# cv.split(train_features, train_labels)
for i, (train, test) in enumerate(cv.split(features,labels)):# cv.split(train_features, train_labels)
    rf.fit(features[train], labels[train])
    viz = plot_roc_curve(rf, features[test], labels[test],
                         alpha=0.0, lw=1, ax=ax,name='ROC fold {}'.format(i))
    interp_tpr = np.interp(mean_fpr, viz.fpr, viz.tpr)
    precision_score_list.append(precision_score(df.loc[test]['classification'].to_list(),[rf.predict(np.array(df.drop(['classification','type'], axis = 1).loc[j,:]).reshape(1, -1))[0] for j in test], average='binary'))
    recall_score_list.append(recall_score(df.loc[test]['classification'].to_list(),[rf.predict(np.array(df.drop(['classification','type'], axis = 1).loc[j,:]).reshape(1, -1))[0] for j in test], average='binary'))
    f1_score_list.append(f1_score(df.loc[test]['classification'].to_list(),[rf.predict(np.array(df.drop(['classification','type'], axis = 1).loc[j,:]).reshape(1, -1))[0] for j in test], average='binary'))
    accuracy_score_list.append(accuracy_score(df.loc[test]['classification'].to_list(),[rf.predict(np.array(df.drop(['classification','type'], axis = 1).loc[j,:]).reshape(1, -1))[0] for j in test]))
    confusion_matrix_list.append(confusion_matrix(df.loc[test]['classification'].to_list(),[rf.predict(np.array(df.drop(['classification','type'], axis = 1).loc[j,:]).reshape(1, -1))[0] for j in test]).ravel())
    test_index_list.append(test)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    aucs.append(viz.roc_auc)
    joblib.dump(rf,'models/rf_%d.sav'%i)

ax.plot([0, 1], [0, 1], linestyle='--', lw=2, color='grey',
        label='Chance', alpha=.8)
mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)
std_auc = np.std(aucs)
ax.plot(mean_fpr, mean_tpr, color='k',
        label=r'Mean ROC (AUC = %0.2f $\pm$ %0.2f)' % (mean_auc, std_auc),
        lw=4, alpha=.8)
std_tpr = np.std(tprs, axis=0)
tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
ax.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=.2,
                label=r'$\pm$ 1 std. dev.')

ax.set(xlim=[-0.05, 1.05], ylim=[-0.05, 1.05],
       title="ROC StratifiedKFold")
#ax.legend(loc="lower right")
plt.legend(frameon=False)
sns.despine()
sns.set(font_scale=1.3,style='white')
plt.tight_layout()
#plt.savefig('ROC_beta_final.svg',transparent=True)#,dpi=600,)

In [ ]:
len(test_index_list[0])

In [ ]:
#Chossing the model with avarge F1 Score Seems to be rf_
print("ACC\t%.2f,%.2f"%(np.mean(accuracy_score_list),np.std(accuracy_score_list)))#
print(accuracy_score_list)
print("Precision\t%.2f,%.2f"%(np.mean(precision_score_list),np.std(precision_score_list)))#_score_list)
print(precision_score_list)
print("Recall\t%.2f,%.2f"%(np.mean(recall_score_list),np.std(recall_score_list)))#_score_list)
print(recall_score_list)
print("F1\t%.2f,%.2f"%(np.mean(f1_score_list),np.std(f1_score_list)))#
print(f1_score_list)


In [ ]:
test_index = test_index_list[4]
rf = joblib.load('models/rf_4.sav')

In [ ]:
df_short = df.loc[test_index]

In [ ]:
RF_regression = pd.DataFrame()
for i in df_short.index:
    Amyloid_class = df_short.loc[i]['type']
    if Amyloid_class=='Amyloid':
        true_ = 1
    else:
        true_ = 0
    Score =rf.predict(np.array(df.drop(['classification','type'], axis = 1).loc[i,:]).reshape(1, -1))[0]
    RF_regression = RF_regression.append({'Class':Amyloid_class,'Score':Score,'type':true_},ignore_index=True)

In [ ]:
tn, fp, fn, tp = confusion_matrix(RF_regression.type, RF_regression.Score).ravel()
print('TN=%d;FP=%d;FN=%d,TP=%d'%(tn, fp, fn, tp))

In [ ]:
from sklearn.metrics import matthews_corrcoef
print(matthews_corrcoef(RF_regression.type, RF_regression.Score))

In [ ]:
! open .

In [ ]:
fig = plt.figure(figsize=(5, 5))
sns.countplot(x="Score", data=RF_regression,palette=['#B4B5B4','#9D282B'],hue='Class',hue_order=['Non-Amyloid','Amyloid'],lw=2,edgecolor='k')
sns.despine(offset=10, trim=True)
sns.set(font_scale=1.5,style='white')
plt.xlabel('$RF\;classifier\;score$')
plt.legend(frameon=False)
plt.ylabel('$Count$')
plt.tight_layout()
plt.savefig('RF_classifier_test_set.svg',transparent=True)#dpi=1200,

In [ ]:
RF_for_plot

In [ ]:
#TN=160;FP=18;FN=26,TP=76
RF_for_plot = pd.read_csv('/Users/aelazar/PycharmProjects/PhaseSep/classifier/notebooks_for_paper/confusion_matrix_bar_plot.csv',delimiter='\t')
fig = plt.figure(figsize=(5, 5))
#$sns.set_style("whitegrid", )
ax = sns.countplot(x="RF classification",data=RF_for_plot,lw=2,edgecolor='k',hue='Ground truth',palette=['#B4B5B4','#9D282B'])##,hue='Ground truth')#hue='Ground truth',hue_order=['Non-amyloid','Amyloid']
sns.despine(offset=10, trim=True)
sns.set(font_scale=1.5,style='whitegrid',rc={'grid.linestyle': '-','grid.linewidth': 2})

plt.xlabel('$Random\;forest\;classification$')
plt.legend(frameon=False)
plt.ylabel('$Count$')
plt.tight_layout()
plt.savefig('RF_classifier_test_set_v2.svg',transparent=False)#dpi=1200,

In [ ]:
sns.barplot(x='count',hue='Ground truth',data=RF_for_plot)

In [ ]:
joblib.dump(df.drop(['type'],axis=1),'/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/WT_RFR/pkls_RandomForestClassifier_9_9_21/RandomForestClassifier_DataFrame.sav')
joblib.dump(rf,'/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/WT_RFR/pkls_RandomForestClassifier_9_9_21/RandomForestClassifier_model.sav')

In [ ]:
df3 = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/WT_RFR/pkls_RandomForestClassifier_9_9_21/ENST00000000233.pkl')

In [ ]:
len(df3[df3.WT_Windows_above_thershold>=3])

In [ ]:
df3 = pd.read_pickle('/Volumes/aelazar/higher_order_assemblies/classifier/human_ORfs/AF2/WT_RFR/pkls/ENST00000000233_WT.pkl')

In [ ]:
len(df3[df3.WT_Windows_above_thershold>=3])

In [ ]:
from sklearn.model_selection import StratifiedKFold,KFold,ShuffleSplit,StratifiedShuffleSplit,RepeatedStratifiedKFold#https://scikit-learn.org/stable/auto_examples/model_selection/plot_roc_crossval.html
train_features, test_features, train_labels, test_labels = train_test_split(features, labels, test_size = .25, random_state = 42)
cv = StratifiedKFold(n_splits=10)
# cv = KFold(n_splits=10)
rf = RandomForestClassifier(n_estimators = 100, random_state = 0,max_features = 'auto', bootstrap = True)
tprs = []
aucs = []
mean_fpr = np.linspace(0, 1, 100)
fig, ax = plt.subplots(figsize=(7, 7))
precision_score_list = []
recall_score_list  = []
f1_score_list = []
accuracy_score_list =[]
confusion_matrix_list = []
test_index_list = []
for i, (train, test) in enumerate(cv.split(features,labels)):
    print(len(train))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
# Number of features to consider at every split
max_features = ['auto', 'sqrt']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
# Method of selecting samples for training each tree
bootstrap = [True, False]
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}

In [ ]:
random_grid

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
rf = RandomForestRegressor()
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
rf_random.fit(features,labels)

In [ ]:
rf_random.best_params_

In [ ]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search 
param_grid = {
    'bootstrap': [False],
    'max_features': [2, 3],
    'min_samples_split': [1, 2],
    'n_estimators': [550,600,650,700,800,2000]
}
# Create a based model
rf = RandomForestRegressor()
# Instantiate the grid search model
grid_search = GridSearchCV(estimator = rf, param_grid = param_grid, 
                          cv = 3, n_jobs = -1, verbose = 2)

In [ ]:
# Fit the grid search to the data
grid_search.fit(train_features, train_labels)

In [ ]:
grid_search.best_params_